### Install and import packages

In [1]:
import sys
!{sys.executable} -m pip install sqlalchemy pymysql pandas

import pandas as pd
import pymysql
import getpass # May need to import as `from getpass import getpass`
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Enter password and set database name


In [2]:
password = getpass.getpass("Enter your MySQL root password: ")
db_name = "movies"


Enter your MySQL root password:  ········


### Create the movies database

In [3]:
conn = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=password
)
cursor = conn.cursor()
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {db_name}")
conn.commit()
cursor.close()
conn.close()
print(f"Database `{db_name}` is ready.")

Database `movies` is ready.


### Connect to the movies database

In [4]:
url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=password,
    host="127.0.0.1",
    port=3306,
    database=db_name,
)

engine = create_engine(url)

with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE();"))
    print("Connected to database:", result.scalar())

Connected to database: movies


### Create all 6 tables

In [5]:
with engine.connect() as connection:
    connection.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    connection.execute(text("DROP TABLE IF EXISTS production_classification;"))
    connection.execute(text("DROP TABLE IF EXISTS content_description;"))
    connection.execute(text("DROP TABLE IF EXISTS media_resources;"))
    connection.execute(text("DROP TABLE IF EXISTS financial_metrics;"))
    connection.execute(text("DROP TABLE IF EXISTS popularity_metrics;"))
    connection.execute(text("DROP TABLE IF EXISTS general_information;"))
    connection.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

    connection.execute(text("""
        CREATE TABLE general_information (
            id INT PRIMARY KEY,
            title VARCHAR(255),
            original_title VARCHAR(255),
            status VARCHAR(50),
            release_date DATE,
            runtime INT,
            adult BOOLEAN,
            original_language VARCHAR(10)
        );
    """))

    connection.execute(text("""
        CREATE TABLE popularity_metrics (
            id INT PRIMARY KEY,
            vote_average FLOAT,
            vote_count INT,
            popularity FLOAT,
            FOREIGN KEY (id) REFERENCES general_information(id)
        );
    """))

    connection.execute(text("""
        CREATE TABLE financial_metrics (
            id INT PRIMARY KEY,
            budget INT,
            revenue INT,
            FOREIGN KEY (id) REFERENCES general_information(id)
        );
    """))

    connection.execute(text("""
        CREATE TABLE media_resources (
            id INT PRIMARY KEY,
            homepage VARCHAR(500),
            imdb_id VARCHAR(20),
            poster_path VARCHAR(255),
            backdrop_path VARCHAR(255),
            FOREIGN KEY (id) REFERENCES general_information(id)
        );
    """))

    connection.execute(text("""
        CREATE TABLE content_description (
            id INT PRIMARY KEY,
            overview TEXT,
            tagline VARCHAR(500),
            keywords TEXT,
            FOREIGN KEY (id) REFERENCES general_information(id)
        );
    """))

    connection.execute(text("""
        CREATE TABLE production_classification (
            id INT PRIMARY KEY,
            genres TEXT,
            production_companies TEXT,
            production_countries TEXT,
            spoken_languages TEXT,
            FOREIGN KEY (id) REFERENCES general_information(id)
        );
    """))

    connection.commit()

print("All 6 tables created.")

All 6 tables created.


### Load the CSV

In [6]:
csv_path = "TMDB_movie_dataset_25k_samples.csv"

df = pd.read_csv(csv_path)
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce').dt.strftime('%Y-%m-%d')
df['adult'] = df['adult'].apply(lambda x: 1 if x is True else 0)
df = df.where(pd.notnull(df), None)

print(f"Loaded {len(df)} rows from CSV.")
df.head(3)

Loaded 25000 rows from CSV.


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,670014,The Fighting Cheat,0.0,0,Released,1926-02-11,0,0,0,NaN,...,The Fighting Cheat,A drifter befriends wounded outlaw Lafe Wells....,0.933,NaN,NaN,Western,Action Pictures,United States of America,NaN,NaN
1,503584,Eaux profondes,0.0,0,Released,2012-02-07,0,2,0,NaN,...,Eaux profondes,The loneliness of the white fish in the goldfi...,0.600,/5vZrvrpvWLICUet6sWQY9VHHIJB.jpg,NaN,NaN,NaN,NaN,NaN,NaN
2,1146817,Dark Road,0.0,0,Released,NaN,0,90,0,NaN,...,Dark Road,The Devil comes home to Missouri. On a foggy n...,0.600,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Insert data into tables

In [7]:
df[['id','title','original_title','status','release_date','runtime','adult','original_language']] \
    .to_sql('general_information', con=engine, if_exists='append', index=False)

df[['id','vote_average','vote_count','popularity']] \
    .to_sql('popularity_metrics', con=engine, if_exists='append', index=False)

df[['id','budget','revenue']] \
    .to_sql('financial_metrics', con=engine, if_exists='append', index=False)

df[['id','homepage','imdb_id','poster_path','backdrop_path']] \
    .to_sql('media_resources', con=engine, if_exists='append', index=False)

df[['id','overview','tagline','keywords']] \
    .to_sql('content_description', con=engine, if_exists='append', index=False)

df[['id','genres','production_companies','production_countries','spoken_languages']] \
    .to_sql('production_classification', con=engine, if_exists='append', index=False)

print("All data inserted.")

All data inserted.


### Check row counts

In [8]:
tables = ['general_information','popularity_metrics','financial_metrics',
          'media_resources','content_description','production_classification']

for table in tables:
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table};", con=engine)
    print(f"{table}: {count['n'][0]} rows")

general_information: 25000 rows
popularity_metrics: 25000 rows
financial_metrics: 25000 rows
media_resources: 25000 rows
content_description: 25000 rows
production_classification: 25000 rows


### Analytical Questions and Queries

Question 1: Does a higher production budget lead to better outcomes?

In [9]:
pd.read_sql("""
        SELECT 
            general_information.title,
            financial_metrics.budget,
            financial_metrics.revenue,
            ROUND(
                CAST(financial_metrics.revenue AS FLOAT) / NULLIF(financial_metrics.budget, 0),
                2
            ) AS roi
        FROM financial_metrics
        JOIN general_information
            ON financial_metrics.id = general_information.id
        WHERE financial_metrics.budget > 0
          AND financial_metrics.revenue > 0
        ORDER BY financial_metrics.budget DESC;
    """, con=engine)

    

,title,budget,revenue,roi
0,Black Panther: Wakanda Forever,250000000,859102154,3.44
1,The Amazing Spider-Man,215000000,757930663,3.53
2,The Jungle Book,175000000,966550600,5.52
3,Poseidon,160000000,181674817,1.14
4,Final Fantasy: The Spirits Within,137000000,85131830,0.62
...,...,...,...,...
308,Er Conde Suelto In Hollywood,1,2,2.00
309,bay789stream,1,1,1.00
310,Пираты карибского моря и банка огурцов,1,1,1.00
311,AFTHOTWTTH,1,1,1.00


Question 2: Does higher audience engagement lead to higher revenue?

In [11]:
pd.read_sql("""
        SELECT
            general_information.title,
            popularity_metrics.vote_count,
            popularity_metrics.vote_average,
            popularity_metrics.popularity,
            financial_metrics.revenue
        FROM popularity_metrics popularity_metrics
        JOIN financial_metrics 
            ON popularity_metrics.id = financial_metrics.id
        JOIN general_information 
            ON general_information.id = popularity_metrics.id
        WHERE financial_metrics.revenue > 0
          AND popularity_metrics.vote_count > 0
        ORDER BY popularity_metrics.vote_count DESC;
    """, con=engine)

    

,title,vote_count,vote_average,popularity,revenue
0,The Truman Show,16827,8.133,42.9540,264118201
1,The Amazing Spider-Man,16352,6.691,78.9890,757930663
2,Hacksaw Ridge,12668,8.195,74.9940,175302354
3,Zombieland,11466,7.300,32.0800,102391540
4,The Hangover Part II,9789,6.468,51.8080,586764305
...,...,...,...,...,...
290,SIRBU E BUGHI,1,10.000,0.0857,12
291,HeartLink,1,10.000,0.1857,14
292,LIL WAYNE LIVE Rolling Loud Cali 2023,1,10.000,0.1214,21325600
293,Autemials - Concert for EC Republic Day Festiv...,1,10.000,0.1429,25
